## 4. Forward PINNs

### 4.1 Method and Evaluation

**Question.** If a small neural network is given only the ODE and the initial condition, how well does it predict the population, compared with a standard solver? All data are *synthetic and dimensionless* ($x=N/K$, $\tau=rt$). The domain is $\tau\in[0,T]$ ($T=10$; for Model A, $T$ is the extinction time) and $x\in[0,1]$. The setting below is the **same one used for the inverse problem** (Section 5).

**Equations.** Model A: $\dot x=G(x)-q$; Model B: $\dot x=G(x)-ex$; $G(x)=x(1-x)(x/a-1)$. Write $\dot x=f(x)$.

**Network with a hard initial condition.** A fully connected network $N(s;\theta)$ with layer sizes $(1,32,32,1)$ (JAX, float64) and the $\tanh$ activation, with fixed time scaling $s=\tau/T$, defines

$$\hat x(\tau)=x_0+s\,N(s;\theta),$$

so $\hat x(0)=x_0$ **exactly for any weights**: no initial-condition loss is needed. Automatic differentiation with respect to physical time applies the chain rule $d\hat x/d\tau=T^{-1}\,d\hat x/ds$ (with the product rule on $s\,N$).

**Loss.** At $n=300$ fixed uniform collocation times $\tau_i\in[0,T]$:

$$R(\tau)=\frac{d\hat x}{d\tau}-f(\hat x),\qquad \mathcal L(\theta)=\frac1n\sum_i R(\tau_i)^2 .$$

**Algorithm.** (1) fixed collocation grid; (2) $\hat x$ and $d\hat x/d\tau$ by `jax.grad` + `vmap`; (3) $\mathcal L$; (4) Adam for 1000 iterations at learning rate $10^{-3}$ (about $2/7$ of the 3500-iteration budget) to reach a good region, then L-BFGS for 2500 iterations with the tolerance stop switched off, to converge. No solver solution is used as a label. For A the training interval ends at the extinction time, so the zero tail never enters the residual.

**Evaluation.** Dense DOP853 reference (rtol $10^{-12}$), separate from training. Metrics: RMSE, MAE, global relative $L^2$ (reported in scientific notation), max error, max error where $x_{\rm ref}<0.05$ (near-zero error), negative samples, runtime. A prediction containing NaN or infinity is *never scored* (it is labelled invalid and cannot count as accurate); a run that raises an exception is recorded as failed. A run is *inaccurate* if relative $L^2>5\times10^{-2}$ (threshold fixed beforehand).


In [ ]:
import csv

# Live demo (a few seconds on CPU): short run (100 Adam + 500 L-BFGS), Model B regular, seed 0, final design.
# A short budget is not the final result; compare with the curated table below.
import sys

sys.path.insert(0, "experiments/wp04")
import run as wp04

from mm_allee.pinn.train_forward import evaluate_prediction, predict

case = dict(model="B", a=0.2, harvest=0.3, x0=0.8, tau_end=10.0)
ref = dict(method="DOP853", rtol=1e-12, atol=1e-14, n_points=1001)
tau, x_ref, hit = wp04.reference_trajectory(case, ref)
training = {
    "activation": "tanh",
    "lr": 1e-3,
    "n_collocation": 300,
    "adam_iters": 100,
    "lbfgs_iters": 500,
}
res = wp04.run_pinn(case, 0, {"training": training}, hit)
ev = evaluate_prediction(predict(res.params, res.config, tau), x_ref)
print({k: (f"{v:.3e}" if isinstance(v, float) else v) for k, v in ev.items()})
print(f"{res.wall_seconds:.1f} s, final physics loss {res.history['loss'][-1]:.3e}")

## 4.2 Curated Final Results (Loaded, Not Retrained)


In [ ]:
rows = list(csv.DictReader(open("artifacts/tables/wp04_forward_metrics.csv")))


def fmt(v):
    return f"{float(v):.2e}" if v else "-"


print(
    f"{'case':13s}{'method':7s}{'seed':5s}{'status':12s}{'rel L2':>10s}{'max err':>10s}{'near-0':>10s}{'neg':>5s}{'sec':>7s}"
)
for r in rows:
    print(
        f"{r['case']:13s}{r['method']:7s}{r['seed']:5s}{r['status']:12s}{fmt(r['relative_l2']):>10s}"
        f"{fmt(r['max_abs']):>10s}{fmt(r['max_abs_near_zero']):>10s}{r['boundary_violations']:>5s}{float(r['wall_seconds']):7.2f}"
    )

![Forward PINN vs reference](../artifacts/figures/wp04_forward.png)

![Error and loss](../artifacts/figures/wp04_error_loss.png)

The vertical line in the bottom row marks the Adam to L-BFGS hand-over.

## 4.3 Interpretation

- **Final sweep (tanh, lr $10^{-3}$, seeds 0/1/42):** all 18 runs are accurate, relative $L^2$ from $2.9\times10^{-7}$ to $4.1\times10^{-3}$ (median $6.4\times10^{-6}$); the largest error is A near-fold (collapse after a bottleneck), the hardest case.
- Five runs predict slightly negative populations near zero (B decay seeds 1 and 42: 23 and 61 of 1001 samples; A extinction and A near-fold: 1 sample each). Near-zero error (largest absolute error where $x_{\rm ref}<0.05$): A extinction $8.9\times10^{-8}$ to $1.9\times10^{-7}$, B decay $7.3\times10^{-7}$ to $2.6\times10^{-6}$, A near-fold $4.8\times10^{-6}$ to $8.6\times10^{-3}$.
- In every run L-BFGS stopped at its 2500-iteration limit (not an abnormal exit), so the iteration budget is binding; longer runs were not tried.
- RK45 matches the reference to about $10^{-10}$ in $4$ to $10$ ms; each PINN run took $3.2$ to $4.6$ s. For a scalar ODE the classical solver is the better tool; the PINN's value is as the engine for the inverse problem (Section 5).
- For Model A the horizon $T$ is the extinction time from the reference solver: the forward PINN does **not** predict the extinction time itself.
- *Limitations:* one parameter set per regime, three seeds, setting selected on the same seeds (then confirmed on three others, below), one machine; per-seed results depend on platform and library versions (exact versions are in the manifests). A **small training loss is not evidence of accuracy**: see the diagnostic study in the companion source.


## 4.4 Held-Out Confirmation and Reproducibility

The frozen design (tanh, $(1,32,32,1)$, lr $10^{-3}$, 300 points) was run unchanged on seeds 11, 22, 33, which were **not** used to select it. All 18 runs are accurate (A near-fold $1.8\times10^{-4}$ to $2.3\times10^{-3}$ is the largest); both seed sets give **36 of 36 accurate runs**.

In [ ]:
conf = [
    r
    for r in csv.DictReader(open("artifacts/tables/wp04_confirmation_metrics.csv"))
    if r["method"] == "pinn"
]
for case in dict.fromkeys(r["case"] for r in conf):
    errs = [float(r["relative_l2"]) for r in conf if r["case"] == case]
    print(f"{case:13s} seeds 11/22/33 rel L2: " + ", ".join(f"{e:.1e}" for e in errs))
print("accurate:", sum(float(r["relative_l2"]) <= 5e-2 for r in conf), "of", len(conf))